# Filter AMR papers with LLM

Use the `llm.py` helper to ask a model whether a paper's abstract asserts that a pathogen is showing **significant resistance** to a given antibiotic.


In [21]:

# --- LLM filtering configuration ---
import pandas as pd

from llm import create_llm_client
from amr.filtering import (
    DEFAULT_SYSTEM_PROMPT,
    prepare_pmids_to_review,
    run_llm_filter,
)

SYSTEM_PROMPT = """
You are an expert microbiologist reviewing evidence. Given a title and abstract, decide whether the authors explicitly state or conclude that the specified pathogen is showing clinically meaningful resistance to the specified antibiotic.

Rules:
- Answer "yes" only if the abstract clearly reports resistance, reduced susceptibility, treatment failure due to resistance, or emergence of resistant isolates for that pathogen/antibiotic pair.
- Answer "no" if the abstract does not make that claim, is about a different organism/drug, only mentions testing without resistance, or only provides background.
- Answer "uncertain" if the abstract is missing, ambiguous, or only indirectly related.
- Base the decision only on the provided text; do not infer beyond it.

Return compact JSON with keys: decision (yes/no/uncertain) and rationale (one short sentence).
"""

LLM_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
NCBI_EMAIL = "your_email@example.com"
MAX_PMIDS = 200  # safety cap on loaded pmids before filtering
N_PMIDS_TO_FILTER = 100  # number of PMIDs to process this run after excluding reviewed
RATE_LIMIT = 0.35  # seconds between PubMed calls
OUTPUT_PATH = "pmids_results_filtered.parquet"
LOG_PATH = "pmid_filter.log"
MAX_LLM_CALLS = None  # set to an integer to cap spend per run
DRY_RUN = False  # set True to skip LLM calls
SAVE_EVERY = 50  # checkpoint frequency

llm_kwargs = {}
if MODEL_NAME:
    llm_kwargs["model"] = MODEL_NAME

llm = create_llm_client(provider=LLM_PROVIDER, **llm_kwargs)
print(f"Using provider={LLM_PROVIDER}, model={MODEL_NAME or 'provider default'}; dry_run={DRY_RUN}; max_llm_calls={MAX_LLM_CALLS}")


Using provider=openai, model=gpt-4o-mini; dry_run=False; max_llm_calls=None


In [22]:
#!pip install openai requests pandas lxml

In [23]:

import os
import pandas as pd
from amr.filtering import prepare_pmids_to_review

# Load previous decisions
if os.path.exists(OUTPUT_PATH):
    reviewed_df = pd.read_parquet(OUTPUT_PATH)
    reviewed = set(reviewed_df['pmid'].astype(str)) if not reviewed_df.empty else set()
    print(f"Resuming with {len(reviewed)} PMIDs already reviewed from {OUTPUT_PATH}")
else:
    reviewed_df = None
    reviewed = set()

pmid_df = pd.read_parquet("pmids_results.parquet")
# Exclude reviewed first, THEN cap to the next chunk
long_df = prepare_pmids_to_review(pmid_df, max_pmids=None, exclude_pmids=reviewed)

pmid_total = len(long_df)
reviewed_count = len(reviewed)
print(f"Loaded {pmid_total} candidate PMIDs after exclusion; reviewed set size: {reviewed_count}")

if N_PMIDS_TO_FILTER:
    long_df = long_df.head(N_PMIDS_TO_FILTER)

if len(long_df) == 0:
    print("No PMIDs remain after excluding reviewed. Consider using a fresh OUTPUT_PATH or increasing source data.")

print(f"PMIDs remaining to review this run: {len(long_df)}")
long_df.head()


Resuming with 173 PMIDs already reviewed from pmids_results_filtered.parquet
Loaded 200 candidate PMIDs; reviewed set size: 173
No PMIDs remain after excluding reviewed. Consider using a fresh OUTPUT_PATH or increasing MAX_PMIDS.
Fallback: reprocessing 100 PMIDs.
PMIDs remaining to review this run: 100


,pmid,pathogen,antibiotic,start_date,end_date
0,27544979,Achromobacter denitrificans,Spectinomycin,2015/01/01,2019/12/31
1,27544979,Achromobacter xylosoxidans,Spectinomycin,2015/01/01,2019/12/31
2,15004037,Acinetobacter baumannii,Spectinomycin,2000/01/01,2004/12/31
3,19361944,Acinetobacter baumannii,Spectinomycin,2005/01/01,2009/12/31
4,19028622,Acinetobacter baumannii,Spectinomycin,2005/01/01,2009/12/31


In [24]:

filtered_df = run_llm_filter(
    long_df,
    llm,
    email=NCBI_EMAIL,
    rate_limit=RATE_LIMIT,
    save_every=SAVE_EVERY,
    system_prompt=SYSTEM_PROMPT,
    progress_every=10,
    resume_from=reviewed_df,
    seen_pmids=reviewed,
    output_path=OUTPUT_PATH,
    log_path=LOG_PATH,
    max_llm_calls=MAX_LLM_CALLS,
    dry_run=DRY_RUN,
)
filtered_df.head()


Finished: processed 173 / 100; positives 47 (0.272 fraction).


,pmid,decision,rationale,pathogen,antibiotic,title,abstract,start_date,end_date
0,27544979,no,The abstract discusses Achromobacter xylosoxid...,Achromobacter denitrificans,Spectinomycin,Properties of Achromobacter xylosoxidans highl...,We herein discovered a highly resistant clinic...,2015/01/01,2019/12/31
1,15004037,no,The abstract mentions resistance to spectinomy...,Acinetobacter baumannii,Spectinomycin,Molecular epidemiology of sequential outbreaks...,The molecular epidemiology of multidrug-resist...,2000/01/01,2004/12/31
2,19361944,no,The abstract does not explicitly state resista...,Acinetobacter baumannii,Spectinomycin,Genetic basis of resistance to aminoglycosides...,A total of 75 Acinetobacter isolates resistant...,2005/01/01,2009/12/31
3,19028622,no,The abstract does not mention Acinetobacter ba...,Acinetobacter baumannii,Spectinomycin,Nosocomial spread of armA-mediated high-level ...,"1,310 Enterobacteriaceae, 242 Pseudomonas aeru...",2005/01/01,2009/12/31
4,16000417,yes,"The abstract mentions gene aadA1, which confer...",Acinetobacter baumannii,Spectinomycin,Detection and typing of integrons in epidemic ...,Integrons were sought in Acinetobacter isolate...,2005/01/01,2009/12/31


In [25]:

print("Summary (this run):")
total_processed = len(filtered_df)
positives = (filtered_df['decision'].str.lower() == 'yes').sum()
fraction = positives / total_processed if total_processed else 0
print(f"Processed {total_processed} abstracts; positives {positives} ({fraction:.3f}).")

if os.path.exists(OUTPUT_PATH):
    all_df = pd.read_parquet(OUTPUT_PATH)
    all_total = len(all_df)
    all_pos = (all_df['decision'].str.lower() == 'yes').sum()
    all_fraction = all_pos / all_total if all_total else 0
    print(f"Cumulative: {all_total} abstracts; positives {all_pos} ({all_fraction:.3f}).")

if os.path.exists(LOG_PATH):
    tail = open(LOG_PATH, 'r', encoding='utf-8').read().splitlines()
    tail = tail[-10:] if tail else []
    print("Log tail:")
    for line in tail:
        print(line)


Summary (this run):
Processed 173 abstracts; positives 47 (0.272).
Cumulative: 173 abstracts; positives 47 (0.272).
Log tail:
[86/93] PMID 781182 | Escherichia coli vs Spectinomycin | interval 1975/01/01–1979/12/31
[87/93] PMID 776050 | Escherichia coli vs Spectinomycin | interval 1975/01/01–1979/12/31
[88/93] PMID 131908 | Escherichia coli vs Spectinomycin | interval 1975/01/01–1979/12/31
[89/93] PMID 131199 skipped (no abstract).
[90/93] PMID 130377 | Escherichia coli vs Spectinomycin | interval 1975/01/01–1979/12/31
[91/93] PMID 128555 | Escherichia coli vs Spectinomycin | interval 1975/01/01–1979/12/31
[92/93] PMID 128550 | Escherichia coli vs Spectinomycin | interval 1975/01/01–1979/12/31
[93/93] PMID 129673 | Escherichia coli vs Spectinomycin | interval 1975/01/01–1979/12/31
Finished: processed 173 / 93; positives 47 (0.272 fraction).
Finished: processed 173 / 100; positives 47 (0.272 fraction).


In [26]:

print(f"Saved results to {OUTPUT_PATH}")
filtered_df['decision'].value_counts()


Saved results to pmids_results_filtered.parquet


decision
no           120
yes           47
uncertain      6
Name: count, dtype: int64

In [27]:

# Build positive-only dataset and save
pmid_df_full = pd.read_parquet("pmids_results.parquet")

# Merge decisions onto exploded PMIDs
positive_flags = filtered_df[filtered_df["decision"].str.lower() == "yes"][["pmid"]].copy()
positive_rows = (
    pmid_df_full.explode("pmids")
    .assign(pmid=lambda d: d["pmids"].astype(str).str.extract(r"(\d+)")[0])
    .dropna(subset=["pmid"])
    .merge(positive_flags, on="pmid", how="inner")
)

# Reassemble to original grain
pmids_positive = (
    positive_rows.groupby(["pathogen", "antibiotic", "start_date", "end_date"], as_index=False)
    .agg(
        pmids=("pmid", list),
        count=("pmid", "size"),
        venue_histogram=("venue_histogram", "first"),
    )
)

positive_path = "pmids_results_positive.parquet"
pmids_positive.to_parquet(positive_path, index=False)
print(f"Saved positive-only dataset to {positive_path} with {len(pmids_positive)} rows")
pmids_positive.head()


Saved positive-only dataset to pmids_results_positive.parquet with 350 rows


,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
0,Acinetobacter baumannii,Amikacin,2005/01/01,2009/12/31,[16000417],1,"{""Yonsei medical journal"": 1, ""Journal of micr..."
1,Acinetobacter baumannii,Amikacin,2020/01/01,2024/12/31,[34800708],1,"{""Frontiers in cellular and infection microbio..."
2,Acinetobacter baumannii,Chloramphenicol,2005/01/01,2009/12/31,[16000417],1,"{""Le infezioni in medicina"": 1, ""Current drug ..."
3,Acinetobacter baumannii,Ciprofloxacin,2015/01/01,2019/12/31,[26023211],1,"{""Annals of burns and fire disasters"": 1, ""Jou..."
4,Acinetobacter baumannii,Gentamicin,2005/01/01,2009/12/31,[16000417],1,"{""Yonsei medical journal"": 1, ""Revista Argenti..."


In [28]:
pmids_positive[pmids_positive['count']>2]

,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
239,Escherichia coli,Spectinomycin,1975/01/01,1979/12/31,"[392907, 365758, 82422, 352263, 144865, 141001...",9,"{""Veterinarno-meditsinski nauki"": 1, ""Infectio..."
242,Escherichia coli,Spectinomycin,2005/01/01,2009/12/31,"[19105635, 16436695, 16251283, 16027145]",4,"{""Journal of food science"": 1, ""Foodborne path..."
245,Escherichia coli,Streptomycin,1975/01/01,1979/12/31,"[365758, 82422, 352263, 129673]",4,"{""Journal of food protection"": 1, ""Antimicrobi..."
252,Escherichia coli,Tetracycline,1975/01/01,1979/12/31,"[392907, 365758, 352263]",3,"{""Journal of food protection"": 1, ""Antimicrobi..."


---